# 01 — Attention from Scratch

**Session goal (1–2 hrs):** implement scaled dot-product attention and multi-head attention yourself — first in NumPy, then in PyTorch — and pass every test below.

**Rules for this notebook**
- Write the code **without AI assistance** (turn off Colab's AI completion). Lab coding rounds are unaided; this is the practice.
- Each `TODO` cell is followed by a test cell. Run it: ✅ pass · ❌ wrong answer · ⏳ not implemented yet · 💥 crashed.
- Stuck for more than ~15 minutes on one function? Write down *what* is confusing in `LOG.md`, then look things up. That note is interview material.

**Plan**
1. NumPy: `softmax` → `scaled_dot_product_attention_np`
2. Experiment: *why* divide by √d_k (your optimization lens)
3. PyTorch: batched attention with masking + gradient check
4. PyTorch: `MultiHeadAttention` module, verified against `nn.MultiheadAttention`
5. Self-check questions (say the answers out loud)

In [99]:
# Setup — Colab already has numpy, torch and matplotlib. No installs needed today.
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Colab: {IN_COLAB} | torch {torch.__version__} | numpy {np.__version__} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
torch.manual_seed(0)

Colab: True | torch 2.11.0+cpu | numpy 2.1.3 | device: cpu


In [100]:
# Tiny test runner — run this cell once, no need to read it.
def run_tests(*tests):
    for t in tests:
        name = t.__name__.removeprefix("test_")
        try:
            t()
            print(f"✅ {name}")
        except NotImplementedError:
            print(f"⏳ {name}: not implemented yet")
        except AssertionError as e:
            print(f"❌ {name}: {e}")
        except Exception as e:
            print(f"💥 {name}: {type(e).__name__}: {e}")

## Part 1 — NumPy

### 1a. Softmax

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Two requirements: it must work along any `axis`, and it must be **numerically stable** — `softmax([1000, 1001, 1002])` should not return `nan`. (Hint: softmax is invariant to subtracting a constant. Which constant?)

In [123]:
def softmax(x, axis=-1):
    # TODO: numerically stable softmax along `axis`
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    res = e / np.sum(e, axis=axis,keepdims=True)
    return res

In [124]:
def test_softmax_values():
    out = softmax(np.array([1.0, 2.0, 3.0]))
    expected = np.array([0.09003057, 0.24472847, 0.66524096])
    assert np.allclose(out, expected, atol=1e-6), f"got {out}"

def test_softmax_stable():
    out = softmax(np.array([1000.0, 1001.0, 1002.0]))
    assert not np.isnan(out).any(), "got nan — overflow in exp()"
    assert np.allclose(out, softmax(np.array([0.0, 1.0, 2.0]))), "not shift-invariant"

def test_softmax_axis():
    x = np.random.default_rng(0).standard_normal((4, 5))
    assert np.allclose(softmax(x, axis=-1).sum(-1), 1), "rows don't sum to 1 for axis=-1"
    assert np.allclose(softmax(x, axis=0).sum(0), 1), "columns don't sum to 1 for axis=0"

run_tests(test_softmax_values, test_softmax_stable, test_softmax_axis)

✅ softmax_values
✅ softmax_stable
✅ softmax_axis


### 1b. Scaled dot-product attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Shapes: `Q` is `(..., n_q, d_k)`, `K` is `(..., n_k, d_k)`, `V` is `(..., n_k, d_v)`. Return **both** the output `(..., n_q, d_v)` and the attention weights `(..., n_q, n_k)`.

**Mask convention (used all session):** `mask` is boolean, broadcastable to `(..., n_q, n_k)`. `True` = may attend, `False` = blocked. Blocked positions must get weight exactly 0. (Hint: what value, added before the softmax, becomes 0 after it?)

Use `K.swapaxes(-1, -2)` rather than `K.T` so leading batch dimensions work for free.

In [138]:
def scaled_dot_product_attention_np(Q, K, V, mask=None):
    # TODO:
    #  1. scores = Q K^T / sqrt(d_k)
    qk = Q @ K.swapaxes(-1, -2) / np.sqrt(K.shape[-1])
    #  2. if mask is given, block positions where mask is False
    if mask is not None:
      qk = np.where(mask, qk, -np.inf)
    #  3. weights = softmax over the *keys* axis
    weights = softmax(qk)
    #  4. output = weights @ V
    output = weights @ V
    print(weights.shape)
    print(V.shape)

    return output, weights

In [139]:
def test_attn_hand_example():
    # Worked by hand: scores = [1, 0] / sqrt(2) -> weights ≈ [0.6698, 0.3302]
    Q = np.array([[1.0, 0.0]])
    K = np.array([[1.0, 0.0], [0.0, 1.0]])
    V = np.array([[1.0, 2.0], [3.0, 4.0]])
    out, w = scaled_dot_product_attention_np(Q, K, V)
    assert np.allclose(w, [[0.66976155, 0.33023845]], atol=1e-6), f"weights {w}"
    assert np.allclose(out, [[1.6604769, 2.6604769]], atol=1e-6), f"output {out}"

def test_attn_shapes_and_rows_sum_to_one():
    rng = np.random.default_rng(1)
    Q, K, V = rng.standard_normal((2, 3, 4, 8)), rng.standard_normal((2, 3, 6, 8)), rng.standard_normal((2, 3, 6, 5))
    out, w = scaled_dot_product_attention_np(Q, K, V)
    assert out.shape == (2, 3, 4, 5), f"output shape {out.shape}, expected (2, 3, 4, 5)"
    assert w.shape == (2, 3, 4, 6), f"weights shape {w.shape}, expected (2, 3, 4, 6)"
    assert np.allclose(w.sum(-1), 1), "weights don't sum to 1 over keys — softmax on wrong axis?"

def test_attn_uniform_when_query_is_zero():
    # Zero query -> all scores equal -> uniform weights -> output = mean of V
    V = np.arange(12, dtype=float).reshape(4, 3)
    out, w = scaled_dot_product_attention_np(np.zeros((1, 3)), np.random.default_rng(2).standard_normal((4, 3)), V)
    assert np.allclose(w, 0.25), f"weights {w}"
    assert np.allclose(out, V.mean(0)), "output should be the mean of V"

def test_attn_mask():
    Q = np.array([[1.0, 0.0]]); K = np.array([[1.0, 0.0], [0.0, 1.0]]); V = np.array([[1.0, 2.0], [3.0, 4.0]])
    out, w = scaled_dot_product_attention_np(Q, K, V, mask=np.array([[True, False]]))
    assert np.allclose(w, [[1.0, 0.0]]), f"blocked key should get weight 0, got {w}"
    assert np.allclose(out, V[0]), f"output should equal V[0], got {out}"

run_tests(test_attn_hand_example, test_attn_shapes_and_rows_sum_to_one,
          test_attn_uniform_when_query_is_zero, test_attn_mask)

(1, 2)
(2, 2)
✅ attn_hand_example
(2, 3, 4, 6)
(2, 3, 6, 5)
✅ attn_shapes_and_rows_sum_to_one
(1, 4)
(4, 3)
✅ attn_uniform_when_query_is_zero
(1, 2)
(2, 2)
✅ attn_mask


## Part 2 — Why divide by √d_k? (the optimization lens)

If the entries of `q` and `k` are independent with mean 0 and variance 1, then `q·k` has variance **d_k**. Run the cell below (it uses your `softmax`): it compares attention with and without the scaling as `d_k` grows.

In [140]:
try:
    rng = np.random.default_rng(0)
    n_q, n_k = 256, 32
    print(f"{'d_k':>5} | {'std(q·k)':>9} | {'max weight (unscaled)':>21} | {'max weight (scaled)':>19}")
    for d_k in [4, 16, 64, 256, 1024]:
        Q, K = rng.standard_normal((n_q, d_k)), rng.standard_normal((n_k, d_k))
        raw = Q @ K.T
        w_unscaled = softmax(raw)
        w_scaled = softmax(raw / np.sqrt(d_k))
        print(f"{d_k:>5} | {raw.std():>9.1f} | {w_unscaled.max(-1).mean():>21.3f} | {w_scaled.max(-1).mean():>19.3f}")
except NotImplementedError:
    print("⏳ Implement softmax (Part 1a) first, then re-run this cell.")

  d_k |  std(q·k) | max weight (unscaled) | max weight (scaled)
    4 |       2.0 |                 0.354 |               0.165
   16 |       4.0 |                 0.623 |               0.158
   64 |       7.9 |                 0.825 |               0.168
  256 |      15.5 |                 0.917 |               0.158
 1024 |      32.4 |                 0.952 |               0.172


**Answer in your own words (write it in `LOG.md`):**

1. What happens to the unscaled attention distribution as `d_k` grows?
2. The Jacobian of softmax is $\frac{\partial p}{\partial z} = \text{diag}(p) - pp^\top$. What does it become when one weight is ≈ 1 and the rest ≈ 0? What does that do to gradient flow through attention early in training?
3. So what is the scaling actually buying you — and why exactly √d_k rather than, say, d_k?

*This is a classic interview probe. Your answer to (2) is the one most candidates can't give.*

## Part 3 — PyTorch: batched attention

Port your NumPy function. Inputs are `(B, H, T, d)` tensors (batch, heads, sequence, head dim), same mask convention (`True` = attend). Use `masked_fill` and `torch.softmax`.

The tests compare against PyTorch's built-in `F.scaled_dot_product_attention` and run `torch.autograd.gradcheck` — a numerical check that your backward pass is correct.

In [143]:
def softmax_torch(x, axis=-1):
    # TODO: numerically stable softmax along `axis`
    e = torch.exp(x - torch.amax(x, axis=axis, keepdim=True))
    res = e / torch.sum(e, axis=axis, keepdim=True)
    return res



def scaled_dot_product_attention(q, k, v, mask=None):
    # TODO: same four steps as the NumPy version, in torch. Return (output, weights).
    qk = q @ k.swapaxes(-1, -2) / torch.sqrt(torch.tensor(k.shape[-1]))
    #  2. if mask is given, block positions where mask is False
    if mask is not None:
      qk = torch.where(mask, qk, -np.inf)
    #  3. weights = softmax over the *keys* axis
    weights = softmax_torch(qk)
    #  4. output = weights @ V
    output = weights @ v
    return output, weights

In [144]:
B, H, T, D = 2, 4, 6, 8
q, k, v = (torch.randn(B, H, T, D) for _ in range(3))
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))   # True on and below the diagonal

def test_matches_pytorch_reference():
    out, w = scaled_dot_product_attention(q, k, v)
    ref = F.scaled_dot_product_attention(q, k, v)
    assert out.shape == (B, H, T, D), f"shape {tuple(out.shape)}"
    assert torch.allclose(out, ref, atol=1e-5), f"max diff {(out - ref).abs().max():.2e}"

def test_causal_mask_matches_reference():
    out, w = scaled_dot_product_attention(q, k, v, mask=causal)
    ref = F.scaled_dot_product_attention(q, k, v, attn_mask=causal)
    assert torch.allclose(out, ref, atol=1e-5), f"max diff {(out - ref).abs().max():.2e}"
    assert torch.all(w.triu(diagonal=1) == 0), "a query attended to a future position"

def test_gradcheck():
    args = [torch.randn(1, 2, 4, 3, dtype=torch.double, requires_grad=True) for _ in range(3)]
    ok = torch.autograd.gradcheck(lambda a, b, c: scaled_dot_product_attention(a, b, c, mask=causal[:4, :4])[0], args)
    assert ok, "gradients don't match finite differences"

run_tests(test_matches_pytorch_reference, test_causal_mask_matches_reference, test_gradcheck)

✅ matches_pytorch_reference
✅ causal_mask_matches_reference
✅ gradcheck


## Part 4 — Multi-head attention

Split `d_model` into `n_heads` heads of size `d_head = d_model // n_heads`, attend in each head independently, then concatenate and project.

**Naming convention (the test depends on it):** four `nn.Linear(d_model, d_model)` layers called `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`, with bias (the default).

The reshape is the part people get wrong in interviews: `(B, T, d_model)` → `(B, T, H, d_head)` → `(B, H, T, d_head)`. And back again before `W_o`. (Hint: after `transpose`, you need `.contiguous()` before `.view()` — or use `.reshape()`. Know why.)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model, self.n_heads, self.d_head = d_model, n_heads, d_model // n_heads
        # TODO: create self.W_q, self.W_k, self.W_v, self.W_o
        raise NotImplementedError

    def forward(self, x, mask=None):
        # x: (B, T, d_model); mask: bool, broadcastable to (B, H, T, T), True = attend
        # TODO:
        #  1. project x to q, k, v
        #  2. split heads: (B, T, d_model) -> (B, H, T, d_head)
        #  3. attention with your scaled_dot_product_attention
        #  4. merge heads: (B, H, T, d_head) -> (B, T, d_model)
        #  5. output projection W_o
        raise NotImplementedError

In [ ]:
d_model, n_heads, B, T = 32, 4, 2, 7
x = torch.randn(B, T, d_model)
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))

def test_mha_shape_and_params():
    m = MultiHeadAttention(d_model, n_heads)
    assert m(x).shape == (B, T, d_model), f"output shape {tuple(m(x).shape)}"
    n = sum(p.numel() for p in m.parameters())
    assert n == 4 * (d_model * d_model + d_model), f"{n} parameters, expected {4 * (d_model**2 + d_model)}"

def test_mha_matches_pytorch():
    torch.manual_seed(0)
    m = MultiHeadAttention(d_model, n_heads).eval()
    ref = nn.MultiheadAttention(d_model, n_heads, batch_first=True).eval()
    with torch.no_grad():  # copy your weights into PyTorch's implementation
        ref.in_proj_weight.copy_(torch.cat([m.W_q.weight, m.W_k.weight, m.W_v.weight]))
        ref.in_proj_bias.copy_(torch.cat([m.W_q.bias, m.W_k.bias, m.W_v.bias]))
        ref.out_proj.weight.copy_(m.W_o.weight)
        ref.out_proj.bias.copy_(m.W_o.bias)
    out = m(x, mask=causal)
    ref_out, _ = ref(x, x, x, attn_mask=~causal, need_weights=False)  # PyTorch's mask: True = BLOCK
    assert torch.allclose(out, ref_out, atol=1e-5), f"max diff {(out - ref_out).abs().max():.2e}"

def test_mha_is_causal():
    # Changing the LAST token must not change outputs at earlier positions
    m = MultiHeadAttention(d_model, n_heads).eval()
    x2 = x.clone(); x2[:, -1] = torch.randn(B, d_model)
    a, b = m(x, mask=causal), m(x2, mask=causal)
    assert torch.allclose(a[:, :-1], b[:, :-1], atol=1e-6), "information leaked from the future"
    assert not torch.allclose(a[:, -1], b[:, -1]), "last position should change"

run_tests(test_mha_shape_and_params, test_mha_matches_pytorch, test_mha_is_causal)

## Stretch (optional) — look at the weights

Visualize one head's causal attention pattern. Nothing is trained yet, so the pattern is random — the point is to see the triangular structure the mask enforces. In Week 3–4 you'll come back and compare against a trained model.

In [ ]:
import matplotlib.pyplot as plt
try:
    _, w = scaled_dot_product_attention(*(torch.randn(1, 1, 10, 16) for _ in range(3)),
                                        mask=torch.tril(torch.ones(10, 10, dtype=torch.bool)))
    plt.imshow(w[0, 0].detach(), cmap="viridis"); plt.colorbar()
    plt.xlabel("key position"); plt.ylabel("query position"); plt.title("Causal attention weights (untrained)")
    plt.show()
except NotImplementedError:
    print("⏳ Implement scaled_dot_product_attention (Part 3) first.")

## Wrap-up — self-check (say these out loud, no notes)

1. Walk through the attention formula and every tensor shape for `(B, T, d_model)` input with `H` heads.
2. Why divide by √d_k? *(Use the softmax-Jacobian argument from Part 2.)*
3. What is the time and memory complexity of attention in sequence length `T`? Where does it come from?
4. How does the causal mask work, and why mask with `-inf` rather than multiplying weights by 0 after the softmax?
5. Why multiple heads instead of one head of size `d_model`? Does it add parameters?
6. Why `.contiguous()` before `.view()`?

**Before you close Colab**
- [ ] Add 3 lines to `LOG.md` (did / confused by / interview insight)
- [ ] **File → Save a copy in GitHub** with a meaningful commit message

**Next session (Week 3):** a full transformer block — pre-norm (LayerNorm/RMSNorm), MLP, residuals — and overfitting a tiny dataset to prove it all works.